# 88 — E3FP / 3D Conformer Fingerprints

2D fingerprints (ECFP) ignore 3D shape. PXR has a large, flexible LBD that is shape-selective.

Strategy:
1. Generate 3D conformers with RDKit ETKDG (10 conformers, prune RMSD=0.5, pick lowest-energy)
2. Compute shape descriptors: PMI ratios, NPR, Asphericity, Eccentricity, SpherocityIndex
3. Compute USRCAT fingerprint (ultrafast shape + pharmacophore recognition)
4. Combine with Morgan+RDKit → LGBM

3D descriptors capture the 'globularity vs planarity vs rod-like' shape spectrum that ECFP misses.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors3D, rdMolDescriptors
from rdkit.Chem import rdDistGeom

def generate_conformer(smi, n_confs=10, max_attempts=200, seed=42):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    mol = Chem.AddHs(mol)
    params = rdDistGeom.ETKDGv3()
    params.randomSeed = seed
    params.pruneRmsThresh = 0.5
    params.numThreads = 1
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
    if not cids: return None
    # Minimize and pick lowest energy
    energies = []
    for cid in cids:
        ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol), confId=cid)
        if ff is None:
            ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
        if ff:
            ff.Minimize(maxIts=200)
            energies.append((ff.CalcEnergy(), cid))
    if energies:
        _, best_cid = min(energies)
        # Set conformer 0 as the best
        mol = Chem.RemoveHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=seed)
        # Re-embed with best params
        mol2 = Chem.MolFromSmiles(smi)
        mol2 = Chem.AddHs(mol2)
        AllChem.EmbedMolecule(mol2, params=params)
        AllChem.MMFFOptimizeMolecule(mol2)
        mol_noh = Chem.RemoveHs(mol2)
        return mol_noh
    return None

def shape_descriptors(mol):
    if mol is None or mol.GetNumConformers() == 0: return [np.nan]*12
    try:
        pmi1 = Descriptors3D.PMI1(mol)
        pmi2 = Descriptors3D.PMI2(mol)
        pmi3 = Descriptors3D.PMI3(mol)
        npr1 = Descriptors3D.NPR1(mol)
        npr2 = Descriptors3D.NPR2(mol)
        asphericity = Descriptors3D.Asphericity(mol)
        eccentricity = Descriptors3D.Eccentricity(mol)
        spherocity = Descriptors3D.SpherocityIndex(mol)
        inertial = Descriptors3D.InertialShapeFactor(mol)
        gyration = Descriptors3D.RadiusOfGyration(mol)
        # planarity proxy: PMI1/PMI3 (rod=0, sphere=1, disk=0.5)
        rod_like = pmi1/pmi3 if pmi3 > 0 else np.nan
        disc_like = pmi2/pmi3 if pmi3 > 0 else np.nan
        return [pmi1, pmi2, pmi3, npr1, npr2, asphericity,
                eccentricity, spherocity, inertial, gyration, rod_like, disc_like]
    except:
        return [np.nan]*12

print("Generating 3D conformers (this takes a few minutes)...", flush=True)


Generating 3D conformers (this takes a few minutes)...


In [5]:
import multiprocessing as mp

SHAPE_NAMES = ["PMI1","PMI2","PMI3","NPR1","NPR2","Asphericity",
               "Eccentricity","Spherocity","InertialSF","Gyration","Rod","Disc"]

def smiles_to_shape(smi):
    mol = generate_conformer(smi)
    return shape_descriptors(mol)

# Process train
print(f"Processing {len(tr):,} train compounds...", flush=True)
shape_tr = []
for i, smi in enumerate(tr["smiles"].tolist()):
    shape_tr.append(smiles_to_shape(smi))
    if (i+1) % 500 == 0: print(f"  {i+1}/{len(tr)}", flush=True)
X_shape_tr = np.array(shape_tr, dtype=np.float32)
# Impute NaN
col_means = np.nanmean(X_shape_tr, axis=0)
for j in range(X_shape_tr.shape[1]):
    mask = ~np.isfinite(X_shape_tr[:,j])
    X_shape_tr[mask, j] = col_means[j]

print(f"Train shape features: {X_shape_tr.shape}")
print(pd.DataFrame(X_shape_tr, columns=SHAPE_NAMES).describe().round(3).to_string())

# Process test
print(f"\nProcessing {len(te):,} test compounds...", flush=True)
shape_te = []
for i, smi in enumerate(te["smiles"].tolist()):
    shape_te.append(smiles_to_shape(smi))
    if (i+1) % 100 == 0: print(f"  {i+1}/{len(te)}", flush=True)
X_shape_te = np.array(shape_te, dtype=np.float32)
for j in range(X_shape_te.shape[1]):
    mask = ~np.isfinite(X_shape_te[:,j])
    X_shape_te[mask, j] = col_means[j]
print(f"Test shape features: {X_shape_te.shape}")


Processing 4,139 train compounds...


[19:47:46] Molecule does not have explicit Hs. Consider calling AddHs()


[19:47:47] Molecule does not have explicit Hs. Consider calling AddHs()


[19:47:47] Molecule does not have explicit Hs. Consider calling AddHs()


[19:47:48] Molecule does not have explicit Hs. Consider calling AddHs()


...

[20:26:29] Molecule does not have explicit Hs. Consider calling AddHs()


[20:26:29] Molecule does not have explicit Hs. Consider calling AddHs()


[20:26:30] Molecule does not have explicit Hs. Consider calling AddHs()


Test shape features: (513, 12)


[20:26:30] Molecule does not have explicit Hs. Consider calling AddHs()


In [6]:
# Combine 3D shape with combined 2D features
X_3d_tr = np.hstack([X_tr, X_shape_tr])
X_3d_te  = np.hstack([X_te, X_shape_te])

oof_shape = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.train(LGBM, lgb.Dataset(X_3d_tr[tr_idx], label=y_tr[tr_idx]),
                  valid_sets=[lgb.Dataset(X_3d_tr[va_idx], label=y_tr[va_idx])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_shape[va_idx] = m.predict(X_3d_tr[va_idx])
    print(f"  fold {fold+1}  RAE={rae(y_tr[va_idx], oof_shape[va_idx]):.4f}", flush=True)

m_shape = full_metrics(y_tr, oof_shape, cliff_pairs, "combined+3D_shape")
m_shape_a = full_metrics(y_tr[active_mask], oof_shape[active_mask], label="3D [active]")
print("\n" + pd.DataFrame([m_shape, m_shape_a], index=["overall","active"]).round(4).to_string())

m_final = lgb.train(LGBM, lgb.Dataset(X_3d_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(m_final.predict(X_3d_te), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"X_shape_tr.npy", X_shape_tr)
np.save(DATA_PROCESSED/"X_shape_te.npy", X_shape_te)
np.save(DATA_PROCESSED/"oof_3d_shape_conformer.npy", oof_shape)
np.save(DATA_PROCESSED/"te_oof_3d_shape_conformer.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"88_3d_shape_conformer.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


  fold 1  RAE=0.4978


  fold 2  RAE=0.5653


  fold 3  RAE=0.5890


  fold 4  RAE=0.5712


  fold 5  RAE=0.6105


  [combined+3D_shape] RAE=0.5619 MAE=0.5112 R²=0.6072 r=0.7793 ρ=0.7290 τ=0.5363
  [3D [active]] RAE=3.7025 MAE=0.7764 R²=-9.5559 r=0.0841 ρ=0.0923 τ=0.0621

            RAE     MAE      R2  Pearson  Spearman  Kendall
overall  0.5619  0.5112  0.6072   0.7793    0.7290   0.5363
active   3.7025  0.7764 -9.5559   0.0841    0.0923   0.0621


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\88_3d_shape_conformer.csv
Test: min=2.40 med=4.91 max=6.07
